# Training the model for Analysis
In this notebook, we will explain how to train your model with the objective of dataset analysis. Here, we optimize to conserve multiple annotations at the same time (e.g. clonotype and cell type, as a surrogate of Gene Expression). You can determine the influence of both modalities (TCR via clonotype, GEX via cell types) by specifying a weight for annotation. This might require retraining on a couple of weight values for finding a mixture suitable for your analysis.

In [1]:
# comet-ml must be imported before torch and sklearn
import comet_ml
import scanpy as sc

import mvtcr.utils_training as utils
utils.fix_seeds(42) #This sets torch.manual_seed, np.random.seed, random.seed for reproducibility; optuna sampler seed can be set later

2025-09-08 13:42:39.797368: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-08 13:42:39.811845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757353359.829985 1440186 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757353359.835433 1440186 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757353359.848899 1440186 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Data Preperation
First we load the data we just created (01_preprocessing notebook) via the Scanpy API. The train-val split is already there.

In [2]:
adata = sc.read_h5ad('v7_avidity.h5ad')
adata.obs['set'].value_counts()

set
train    30966
val      10241
test     10196
Name: count, dtype: int64

In [6]:
adata

AnnData object with n_obs × n_vars = 51403 × 5000
    obs: 'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len', 'has_binding', 'set'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'aa_to_id', 'chain_indices', 'clonotype', 'hvg', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

## Defining the model parameters
We need to proivde the model a couple of parameters:
- study_name: Name for logging
- comet_workspace: we logged some of the experiments via Comet-ML. This gives you more information on the training process, but is not needed. We will therefore use None here. Otherwise, specifiy the workspace name of your Comet-ML project.
- model_name: assigns which model is used from (rna, tcr, moe, poe, concat). We will use the best performing moe.
- balanced_sample: oversample rare elements of this column. Recommended to use a column storing the clonotype to avoid overfitting.
- metadata: annotation to color the umaps when storing immediate results on Comet-ML. If no Comet-ML is used, pass an empty list
- save_path: path to store the trained models over multiple training runs
- conditional: name of a conditional variable (see preprocessing). The model partially removes batch effects over this column.
- n_epoch: amounts of epochs to train the model. For the paper we used 200 epochs. For showcasing however, we will reduce it to 5.

In [3]:
params_experiment = {
    'study_name': 'haniffa_tutorial',
    'comet_workspace': None, 
    'model_name': 'moe',
    'balanced_sampling': 'clonotype',
    'metadata': [],
    'save_path': '../saved_models/haniffa_tutorial',
    'conditional': 'patient_id_ohe',
    'n_epochs': 10,
}

## Defining Optimization parameters
For this analysis, we will optimize to perserve clonotype and cell type ("full_clustering" column in our dataset). This optimization mode is called 'pseudo_metric'. By specifying the weight, we can choose the weighting between both modalities.

In [4]:
params_optimization = {
    'name': 'pseudo_metric',
    'prediction_labels':
        {'clonotype': 1,
         'full_clustering': 1}
}

We also offer the option to exclude cells from evaluation, but not training. To do so, set the value of the prediction label for the desired cells to -99 (in your adata object, not here in the params dict!).

## Calling the training functions
Finally, we need to specificy a couple of parameters for running the training. Training will be aborted either after \<timeout\> seconds or after having trained 3 models with 1 available GPU. Typically, we used considerable larger amount of training runs (e.g. 48 GPU-hours). Note: when increasing the number of GPUs, you will also need to scale the CPU resources, so that the training is not bottlenecked by e.g. dataloading on CPU. The argument sampler_seed can be specified make the optuna sampler behave deterministically. However, for fully reproducible results training should be run sequentially https://optuna.readthedocs.io/en/stable/faq.html.  

In [5]:
from mvtcr.models.model_selection import run_model_selection

timeout = (20*60)
n_samples = 3
n_gpus = 1
seed = 42
run_model_selection(adata, params_experiment, params_optimization, n_samples, timeout, n_gpus, sampler_seed=seed)

[I 2025-09-08 13:42:57,029] A new study created in RDB with name: haniffa_tutorial
[W 2025-09-08 13:42:57,777] Trial 0 failed with parameters: {'dropout': 0.1, 'activation': 'linear', 'rna_hidden': 1500, 'hdim': 200, 'shared_hidden': 100, 'rna_num_layers': 1, 'tfmr_encoding_layers': 4, 'loss_weights_kl': 4.0428727350273357e-07, 'loss_weights_tcr': 0.034702669886504146, 'lr': 1.0994335574766187e-05, 'zdim': 50, 'tfmr_embedding_size': 16, 'tfmr_num_heads': 8, 'tfmr_dropout': 0.15000000000000002} because of the following error: KeyError('patient_id_ohe').
Traceback (most recent call last):
  File "/ihome/ylee/yiz133/.local/lib/python3.11/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/ihome/ylee/yiz133/Code/mvTCR/mvTCR/mvtcr/models/model_selection.py", line 152, in <lambda>
    study.optimize(lambda trial: objective(trial, adata, suggest_params, params_experiment, params_optimization),
         

KeyError: 'patient_id_ohe'

### Optuna Dashboard
Optionally you can use Optuna Dashboard to get more insights of e.g. how certain hyperparameters influence model selection. Find more information on how to use it here: https://optuna-dashboard.readthedocs.io/en/latest/getting-started.html

## Output
The console output indicates the best model after Hyperparameter Optimization. We will now load this model and embedd our data with it. Following, we can continue with standard analysis.

In [ ]:
path_model = '../saved_models/haniffa_tutorial/trial_2/best_model_by_metric.pt'
model = utils.load_model(adata, path_model)

The data is embedded and the cell annotation is copied to the resulting AnnData object.

In [ ]:
latent_moe = model.get_latent(adata, metadata=[], return_mean=True, copy_adata_obs=True)

We can now visualize the data via UMAPs.

In [ ]:
sc.pp.neighbors(latent_moe, use_rep='X')
sc.tl.umap(latent_moe)
sc.pl.umap(latent_moe, color=['full_clustering', 'sample_id'])

Note, that the results here show severe batch-effect and bad conservence of cell type. This is due to the limited amount of patients used, the low amount of epochs per training, and the limited runs for HPO. However, these values where choosen in order to have  short computation times for this tutorial.